# Predicción de procesos de compra sin adjudicación

## Entrega 2: descripción, exploración y preparación del dataset

El proyecto utiliza datos públicos de procesos de compra del sistema Buenos Aires Compras. Los archivos fueron descargados el 29 de octubre de 2026 y se encuentran organizados siguiendo la estructura de Cookiecutter Data Science.

El objetivo de esta etapa es conocer la estructura de los datos, identificar los tipos de variables, revisar valores faltantes y duplicados, y preparar una observación por cada proceso de compra antes de generar el modelo de aprendizaje automático.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Carga de los datos

Para el análisis se utilizan cuatro archivos relacionados. `main.csv` contiene la información principal de los procesos de compra, mientras que los demás archivos permiten conocer los renglones solicitados y cuáles fueron adjudicados.

In [3]:
procesos = pd.read_csv("../data/raw/main.csv")
adjudicaciones = pd.read_csv("../data/raw/awards.csv")
items_solicitados = pd.read_csv("../data/raw/tender_items.csv")
items_adjudicados = pd.read_csv("../data/raw/awards_items.csv")

In [4]:
procesos.head()

,_link,id,tag,date,ocid,language,initiationType,tender_id,tender_additionalProcurementCategories,tender_title,...,tender_enquiryPeriod_startDate,tender_enquiryPeriod_durationInDays,tender_procuringEntity_id,tender_procuringEntity_name,tender_techniques_frameworkAgreement_method,tender_techniques_frameworkAgreement_value_amount,tender_techniques_frameworkAgreement_value_currency,tender_techniques_frameworkAgreement_period_endDate,tender_techniques_frameworkAgreement_period_startDate,tender_techniques_frameworkAgreement_period_durationInDays
0,id-0.0,ocds-bulbcf-10002-0591-LPU22-3-0-2022-05-24T10...,compiled,2022-05-24T10:15:00-03:00,ocds-bulbcf-10002-0591-LPU22-3-0,es,tender,10002-0591-LPU22,Concesiones,Licitación Pública Bibliotecas de la Ciudad,...,2022-05-24T10:16:00-03:00,36,CABA-UE-10002,Direccion General Concesiones y Permisos,NaN,NaN,NaN,NaN,NaN,NaN
1,id-0.1,ocds-bulbcf-101-0316-CDI22-1-1-2022-04-05T12:0...,compiled,2022-04-05T12:00:00-03:00,ocds-bulbcf-101-0316-CDI22-1-1,es,tender,101-0316-CDI22,Servicios,Servicio para la realización de talleres a fin...,...,2022-04-05T12:30:00-03:00,1,CABA-UE-101,SECRETARIA LEGAL Y TECNICA,NaN,NaN,NaN,NaN,NaN,NaN
2,id-0.2,ocds-bulbcf-101-1835-LPU21-4-5-2022-02-07T12:0...,compiled,2022-02-07T12:00:00-03:00,ocds-bulbcf-101-1835-LPU21-4-5,es,tender,101-1835-LPU21,Equipamiento para Oficina,Adquisición de papel,...,2022-02-07T12:30:00-03:00,3,CABA-UE-101,SECRETARIA LEGAL Y TECNICA,NaN,NaN,NaN,NaN,NaN,NaN
3,id-0.3,ocds-bulbcf-112-0272-CME22-3-0-2022-03-31T14:0...,compiled,2022-03-31T14:00:00-03:00,ocds-bulbcf-112-0272-CME22-3-0,es,tender,112-0272-CME22,Seguridad,Provisión de servicios relacionados con el Si...,...,2022-03-31T15:00:00-03:00,7,CABA-UE-112,"CONSEJO DE LOS DERECHOS DE LAS NIÑAS, NIÑOS Y ...",NaN,NaN,NaN,NaN,NaN,NaN
4,id-0.4,ocds-bulbcf-112-0960-CME22-5-1-2022-04-26T14:0...,compiled,2022-04-26T14:00:00-03:00,ocds-bulbcf-112-0960-CME22-5-1,es,tender,112-0960-CME22,Construcción,Adquisicón de herramientas varias,...,2022-04-26T14:30:00-03:00,2,CABA-UE-112,"CONSEJO DE LOS DERECHOS DE LAS NIÑAS, NIÑOS Y ...",NaN,NaN,NaN,NaN,NaN,NaN


## 2. Dimensiones de los archivos

A continuación se observa la cantidad de filas y columnas de cada archivo utilizado. Las filas representan registros y las columnas representan las características disponibles.

In [5]:
print("main.csv:", procesos.shape)
print("awards.csv:", adjudicaciones.shape)
print("tender_items.csv:", items_solicitados.shape)
print("awards_items.csv:", items_adjudicados.shape)

main.csv: (23298, 33)
awards.csv: (13835, 12)
tender_items.csv: (23298, 11)
awards_items.csv: (13835, 12)


In [6]:
procesos.info()

<class 'pandas.DataFrame'>
RangeIndex: 23298 entries, 0 to 23297
Data columns (total 33 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   _link                                                       23298 non-null  str    
 1   id                                                          23298 non-null  str    
 2   tag                                                         23298 non-null  str    
 3   date                                                        23298 non-null  str    
 4   ocid                                                        23298 non-null  str    
 5   language                                                    23298 non-null  str    
 6   initiationType                                              23298 non-null  str    
 7   tender_id                                                   23298 non-null  str    
 8   tender_

### Interpretación inicial

El archivo principal contiene 23.298 registros y 33 columnas. Se identifican variables de texto, numéricas y booleanas. Las fechas se encuentran inicialmente almacenadas como texto, por lo que posteriormente deberán convertirse al tipo fecha.

La cantidad de registros todavía no representa necesariamente la cantidad de procesos de compra, ya que un mismo proceso puede contener varios renglones.

In [7]:
valores_faltantes = procesos.isnull().sum()
valores_faltantes[valores_faltantes > 0]

tender_status                                                    36
tender_procuringEntity_name                                      50
tender_techniques_frameworkAgreement_method                   20941
tender_techniques_frameworkAgreement_value_amount             22966
tender_techniques_frameworkAgreement_value_currency           22966
tender_techniques_frameworkAgreement_period_endDate           22966
tender_techniques_frameworkAgreement_period_startDate         23273
tender_techniques_frameworkAgreement_period_durationInDays    23273
dtype: int64

In [8]:
print("Filas duplicadas:", procesos.duplicated().sum())
print("Procesos de compra únicos:", procesos["tender_id"].nunique())

Filas duplicadas: 0
Procesos de compra únicos: 2950
